In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Native C Transpilation of 3-Tier Hierarchical Soft Triage Pipeline (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook transpiles the trained **3-Tier 4-Model Hierarchical LightGBM Triage Model** into **Zero-Dependency Native C Code (`deploy/triage_soft_pipeline.c`)**:

### Transpiled Layer-Exclusive Features & Sub-Models
1. **Layer 1 (8 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_hypertension`, `is_tachypnea`, `is_bradypnea`, `is_tachycardia_total` -> `deploy/lightgbm_layer1_esi1_model.rds`
2. **Layer 2 (18 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_max`, `hr_last_to_max`, `sbp_last_to_max`, `rr_last_to_max` -> `deploy/rf_esi23_esi45_extreme_model.rds`
3. **Layer 3A (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `hr_mean_to_last`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `spo2_mean_to_last`, `rr_mean_to_last` -> `deploy/lightgbm_esi23_model.rds`
4. **Layer 3B (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `is_dyspnea_total`, `hr_mean_to_last` -> `deploy/lightgbm_esi45_model.rds`

Outputs `deploy/triage_soft_pipeline.c` and `deploy/triage_soft_pipeline.h`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Export LightGBM Trees & Transpile to C Code
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(lightgbm)
})
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
l1_obj  <- readRDS(file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
l2_obj  <- readRDS(file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
l3a_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi23_model.rds"))
l3b_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi45_model.rds"))
l1_model  <- if (is.list(l1_obj)  && "model" %in% names(l1_obj))  l1_obj$model  else l1_obj
l2_model  <- if (is.list(l2_obj)  && "model" %in% names(l2_obj))  l2_obj$model  else l2_obj
l3a_model <- if (is.list(l3a_obj) && "model" %in% names(l3a_obj)) l3a_obj$model else l3a_obj
l3b_model <- if (is.list(l3b_obj) && "model" %in% names(l3b_obj)) l3b_obj$model else l3b_obj
preproc <- if (is.list(l1_obj) && "preproc" %in% names(l1_obj)) l1_obj$preproc else NULL
dt_l1  <- lgb.model.dt.tree(l1_model)
dt_l2  <- lgb.model.dt.tree(l2_model)
dt_l3a <- lgb.model.dt.tree(l3a_model)
dt_l3b <- lgb.model.dt.tree(l3b_model)
cat(sprintf("Loaded Trees for C Transpilation: L1=%d trees, L2=%d trees, L3A=%d trees, L3B=%d trees\n",
            max(dt_l1$tree_index)+1, max(dt_l2$tree_index)+1, max(dt_l3a$tree_index)+1, max(dt_l3b$tree_index)+1))

In [ ]:
# ---------------------------------------------------------
# Step 2: Write Native C Header & Source Files
# ---------------------------------------------------------
import os
deploy_dir = "../deploy"
if not os.path.exists(deploy_dir): deploy_dir = "deploy"
os.makedirs(deploy_dir, exist_ok=True)
c_header_content = """/* triage_soft_pipeline.h - 3-Tier Hierarchical LightGBM Triage Pipeline */
#ifndef TRIAGE_SOFT_PIPELINE_H
#ifndef TRIAGE_SOFT_PIPELINE_H
#define TRIAGE_SOFT_PIPELINE_H
#ifdef __cplusplus
extern "C" {
#endif
typedef struct {
    float age;
    float gender;
    float cc_breathingdifficulty;
    float triage_vital_hr;
    float triage_vital_sbp;
    float triage_vital_rr;
    float triage_vital_o2;
    float pulse_last;
    float resp_last;
    float spo2_last;
    float sbp_last;
    float pulse_min;
    float resp_min;
    float spo2_min;
    float sbp_min;
    float pulse_max;
    float resp_max;
    float spo2_max;
    float sbp_max;
} TriageInput;
typedef struct {
    float p_esi1;
    float p_esi2;
    float p_esi3;
    float p_esi4;
    float p_esi5;
    int predicted_esi;
} TriageResult;
void predict_triage_soft(const TriageInput* in, TriageResult* out);
#ifdef __cplusplus
}
#endif
#endif /* TRIAGE_SOFT_PIPELINE_H */
"""
with open(os.path.join(deploy_dir, "triage_soft_pipeline.h"), "w") as f:
    f.write(c_header_content)
print(f"Header exported to: {os.path.join(deploy_dir, 'triage_soft_pipeline.h')}")